# 02. Gemini による映画ストーリー推論

Notebook 01 で決定した重みを使って類似度を計算し、上位20件の候補を Gemini に渡して元の映画ペアを推論させる。

In [1]:
import gc
import re
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sudachipy import dictionary, tokenizer
from tqdm.notebook import tqdm

In [11]:
# 埋め込みモデル
MODEL_NAMES = [
    "Qwen/Qwen3-Embedding-0.6B",
    "cl-nagoya/ruri-v3-310m",
    "stsb-xlm-r-multilingual",
]

# Notebook 01 で決定した重み
WEIGHTS = {
    "Qwen3-Embedding-0.6B": 0.2,
    "ruri-v3-310m": 0.2,
    "stsb-xlm-r-multilingual": 0.1,
    "tfidf": 0.5,
}

# Gemini 設定
GEMINI_MODEL = "gemini-3-flash-preview"
TOP_K = 20
API_SLEEP = 0.5

In [3]:
# Gemini クライアント初期化
load_dotenv("../.env")
client = genai.Client()

## データ読み込み

In [4]:
base_df = pd.read_csv("../dataset/base_stories.tsv", sep="\t")
practice_df = pd.read_csv("../dataset/fiction_stories_practice.tsv", sep="\t")
test_df = pd.read_csv("../dataset/fiction_stories_test.tsv", sep="\t")

base_stories = base_df["story"].tolist()

print(f"Base: {len(base_df)}, Practice: {len(practice_df)}, Test: {len(test_df)}")

Base: 50, Practice: 20, Test: 340


## 類似度計算

Notebook 01 と同じ手順で統合類似度行列を算出する。

In [5]:
class TextProcessor:
    def __init__(self):
        self.tokenizer_obj = dictionary.Dictionary().create()
        self.mode = tokenizer.Tokenizer.SplitMode.C

    def tokenize(self, text: str) -> str:
        tokens = self.tokenizer_obj.tokenize(text, self.mode)
        words = [
            token.normalized_form()
            for token in tokens
            if token.part_of_speech()[0] in ["名詞", "動詞", "形容詞"]
        ]
        return " ".join(words)


text_processor = TextProcessor()

In [6]:
def compute_combined_similarity(target_stories: list[str]) -> np.ndarray:
    """複数モデル + TF-IDF の加重平均で統合類似度行列を算出する"""
    similarities = {}

    # SentenceTransformer
    for model_name in MODEL_NAMES:
        print(f"Processing: {model_name}")
        model = SentenceTransformer(model_name)
        base_emb = model.encode(base_stories, show_progress_bar=True)
        target_emb = model.encode(target_stories, show_progress_bar=True)
        key = model_name.split("/")[-1]
        similarities[key] = cosine_similarity(target_emb, base_emb)
        del model
        gc.collect()

    # TF-IDF
    print("Processing: TF-IDF")
    base_tok = [text_processor.tokenize(t) for t in tqdm(base_stories, desc="Base")]
    target_tok = [text_processor.tokenize(t) for t in tqdm(target_stories, desc="Target")]
    vectorizer = TfidfVectorizer()
    tfidf_all = vectorizer.fit_transform(base_tok + target_tok)
    similarities["tfidf"] = cosine_similarity(tfidf_all[len(base_tok):], tfidf_all[:len(base_tok)])

    # MinMaxスケーリング + 加重平均
    combined = np.zeros_like(list(similarities.values())[0])
    for key, sim in similarities.items():
        scaler = MinMaxScaler()
        flat = sim.flatten().reshape(-1, 1)
        sim_scaled = scaler.fit_transform(flat).reshape(sim.shape)
        combined += WEIGHTS[key] * sim_scaled

    return combined

## 候補抽出 & プロンプト生成

In [7]:
def extract_top_candidates(combined_sim: np.ndarray) -> pd.DataFrame:
    """各ストーリーの上位 TOP_K 件を展開した DataFrame を返す"""
    rows = []
    for i in range(combined_sim.shape[0]):
        ranked_idx = np.argsort(combined_sim[i])[::-1][:TOP_K]
        for rank, idx in enumerate(ranked_idx, 1):
            base_row = base_df.iloc[idx]
            rows.append({
                "story_id": i,
                "rank": rank,
                "title": base_row["title"],
                "movie_id": base_row["id"],
                "story": base_row["story"],
            })
    return pd.DataFrame(rows)


def generate_prompt(story_id: int, target_story: str, candidates_df: pd.DataFrame) -> str:
    """2つの映画を同時に特定させるプロンプトを生成する"""
    cands = candidates_df[candidates_df["story_id"] == story_id].sort_values("rank")
    candidates_text = "\n\n".join(
        f"{row['movie_id']}.{row['title']}\nストーリー:\n{row['story']}"
        for _, row in cands.iterrows()
    )

    return f"""以下の「合成ストーリー」は、2つの映画のあらすじを混ぜ合わせて作られたものです。
「合成ストーリー」を分析し、元となった２つの映画を候補映画一覧から特定してください。
ただし、出力は番号のみ（例:1,2）で半角数字とカンマだけの構成とすること、理由や根拠は絶対に出力しないでください。

【合成ストーリー】
{target_story}

【候補映画一覧】
{candidates_text}"""

## Gemini 推論

In [8]:
def run_inference(target_df: pd.DataFrame, combined_sim: np.ndarray) -> pd.DataFrame:
    """Gemini で推論を実行し、予測結果の DataFrame を返す"""
    candidates_df = extract_top_candidates(combined_sim)
    predictions = []

    for sid in tqdm(range(len(target_df)), desc="Gemini推論"):
        prompt = generate_prompt(sid, target_df.iloc[sid]["story"], candidates_df)
        response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        output = response.text.strip()

        cleaned = re.sub(r"[^\d,]", "", output)
        numbers = [n for n in cleaned.split(",") if n]

        if len(numbers) >= 2:
            pred_a, pred_b = int(numbers[0]), int(numbers[1])
        else:
            pred_a = pred_b = None
            print(f"[{sid}] パース失敗: {output}")

        predictions.append({
            "story_id": sid,
            "predicted_a": pred_a,
            "predicted_b": pred_b,
            "raw_output": output,
        })
        time.sleep(API_SLEEP)

    return pd.DataFrame(predictions)

## Practice 精度検証

In [9]:
practice_sim = compute_combined_similarity(practice_df["story"].tolist())

Processing: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/home/sakamoto/competition/signate/movie_story/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Processing: TF-IDF


Base:   0%|          | 0/50 [00:00<?, ?it/s]

Target:   0%|          | 0/20 [00:00<?, ?it/s]

In [12]:
practice_pred = run_inference(practice_df, practice_sim)

Gemini推論:   0%|          | 0/20 [00:00<?, ?it/s]

In [13]:
# 精度検証
correct = 0
for _, pred in practice_pred.iterrows():
    sid = pred["story_id"]
    actual = {int(practice_df.iloc[sid]["id_a"]), int(practice_df.iloc[sid]["id_b"])}
    predicted = {pred["predicted_a"], pred["predicted_b"]} - {None}

    if predicted == actual:
        correct += 1
    else:
        print(f"Story #{sid}: 正解={actual}, 予測={predicted}")

print(f"\nAccuracy: {correct}/{len(practice_pred)} ({correct / len(practice_pred) * 100:.1f}%)")

Story #0: 正解={29, 23}, 予測={23, 31}
Story #2: 正解={43, 11}, 予測={32, 11}
Story #8: 正解={28, 13}, 予測={10, 28}
Story #9: 正解={26, 30}, 予測={37, 30}
Story #13: 正解={40, 50}, 予測={50, 19}
Story #17: 正解={48, 49}, 予測={48, 19}

Accuracy: 14/20 (70.0%)


## Test 推論

In [ ]:
test_sim = compute_combined_similarity(test_df["story"].tolist())

In [ ]:
test_pred = run_inference(test_df, test_sim)